# CDCN CVPR 2020 · Lượt chạy thử Train/Dev trên OULU-NPU
Chọn **GPU A100** và chạy các cell theo thứ tự. Lượt này dùng 3 video Train và
3 video Dev, mỗi video có 8 khung. Chạy 3 epoch từ khởi tạo mới. Mỗi epoch lấy
ngẫu nhiên một khung mỗi video Train. Dev dùng đủ 8 khung và tính trung bình
các tỷ số tổng depth dự đoán trên tổng map PRNet.

Cell 3 khôi phục ZIP tiền xử lý trên Drive nếu dấu vân tay cấu hình trùng.
Nếu cần dựng từ đầu, cell đó đọc các video được chọn trong TAR và ghi
snapshot lên Drive. Cell 4 lưu checkpoint, điểm video, chỉ số Dev và thời
gian. Chỉ số của 3 video Dev dùng để kiểm tra pipeline. Mốc của paper dùng
toàn bộ Protocol 1 và chấm trên Test.

Nếu đã có kết quả pilot, chạy cell 1–3 rồi chuyển thẳng đến cell 5 để đo
batch trên A100 bằng các khung Train đã có.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-train-pilot")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-train-pilot-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn runtime GPU trước khi tiếp tục")
print("GPU:", torch.cuda.get_device_name(), "| PyTorch:", torch.__version__)

In [ ]:
#@title 2. Nạp code đi kèm
BUNDLE_SHA256 = '8f0e7e0e977f3e630fae6f91d228605297d5af4c9d18bfd2db5280d810151175'
BUNDLE = (
    'H4sIAAAAAAAC/+19+3PbONLg/uyq+R/4cetuyUSi9fArrtHWeRxnJrd5+GJn9va8Kg5FQjbHFMklKccef76//foBkOBDsp1kcrvf'
    'WjUTS3g0gEaj0d1oNPLM3wyESOeeL9zUCzZdN4zDwnWd9OYPX+kzgM/O1hb9hU/z72Cwu1t9x/TheDQe/sEY/OEbfJZ54WXQ/B/+'
    'PT+mab6E6X8F028cH7w0MpELL/MvjNTzL71z4UCB7za+23DdK5HlYRK7rjExzIEzdAaQ8Yenz7/2J2+u/yjJc5F/xdV/7/rf3d0e'
    'N9f/YDh6Wv/f4jPPkoXhuvNlscwErO1wkSZZYXhxnBReAes9x9UvU4sEOMN3G1SHvqvicaynOnGsMubL2EcoXmR4ufEKYX234Ude'
    'nhuHSVxkXl6EV+KlSIuLN0B4Vhw7b5NgGQl7/7sNAz6BmBtqS7JyEc1to/9n410SC1kAP/kyFZllO2VBu8q7FFksohyY1tm0Sv2j'
    'cRSeXxTGMg7/sRTGPMzyop9kgciMIJzPRSZiX+SGlyXLODCKC2H4ArorjDS8FpFjGKcXQoeWZuIqTJY5jjsSCyhLyDPC2I+WgQgA'
    'KtAZwOwXnxIjmc9zUUDh2PCM8fVYh8T9hQkIDMDTIhXU/MKYAT8G9Bow8p6RhxE0Ed0YwTKNQh/ais+NebLMdEhBmAlCfu5UyfME'
    'RnjTM4JrbP2sysCP1R/2jP7Q7nUkDzpT22UHnRAGXUW7G+tsq9nUdL9eQGJtIgnwN5EluTXuGWO7q9wZgptC6WE9F/ByjanGc8IR'
    '/b3uBIAlEUB/2JWdO16aijiw+KfWB6RgJxPnQA0ic2dLJDXLlLXMnuw+kIp/KSvn9tl+jwh+auPiUYsCJvKTlwW0JnpIf0FIk70v'
    'YZyKOE8ygOhl56Kop9Ia0hP2u9YL9VUNqEisAEjcF5OqKYdTgJiKm7SegQnasLkTOD305QuhYZZ7nnkBAHzl+El8NQqsqnxPjQHQ'
    '4gUBLI3JsNWXVn1OvqduJoBNxlApGrq4UVplV3o6XFvnc+/n89APvehh/A7ELa0gzDSUNLApg/krcKJEAjQOXx6+Mw5/Pv5gjAaj'
    'gQGVgP0BI/CToJTbvjIPZaIpiGisOuk3eAkl4UKDpT+Y9owzYgzVd0qe9roqIbP4nEpY73MqUcHh51SSLQ0/p5Jq9fGVFH4eW4nw'
    '06zU/M2rj+d5HiVeMR5pJR7ByuS3J+71OO7liOsUNn/aXfd4/nCTrPjSOcgkaT7ZeyhL+xyAJZ9b5OLBjO5V4nvRA6W4nuFF6YW3'
    'bxCJQacHzmgbuuItFlriyBk8llXRtBPsHn8nkABLptHP1ZQYJedhkXdTYf5wMpQVSpLJeSnpHZ2Bwo1zNQtjL7txfRBYchdlzCS9'
    'cT+FxYXLfbH4T9mJHkxOsCRSmZgxIMas0VYy82ZhFBY3JbPOw/NFEgYSjl64KMt8ugCJ1yp7PUH+osFCUaivJ2hQCK/uelD6pCCk'
    '6neb5iwF8BlIfdhqYTtp8smqJtOGLMCe7SyEF1uSBHEqabOsyPXhHAXm3VvMAs/1ZnlFfUNnUGb4ck/WCXZ7FQV8PX4hcVL1DoZe'
    'Fz6UyMNN2SCxNnoMNTolD7ureoVLJWS4fuDHroZZ7thK/OoY6MxahelabhvdXGQVxkHYeV+TirCr+1J6entyZKQR6GYCVb5+jH9m'
    'oCsZJYqghC4v/d7zV2OqD5lATdJaK02umFO5cT/M/gOr/DwTef7t7L/b28OW/Wc83nmy/3wj+++xyMIEqMZQU09GgjgpxCxJLo1f'
    'EyBbNEdESZLmuE427jcabXCZ1CsuonCmChzDT2VLypczaM+H5krr0kUmPFS5yoRwITY2pHRxLDu3v0qgiLyZiFCDLrxo8o7MJEvI'
    'n5hhIRYokIYxCKpXkDke2Pt1gUHWpe8EQH5HAPKrqg1sQW+JG6mA1+EG0A3cK5qt5YW7TAOvEEpQRSIsBEpvOGhnkQAmkzj0Lbte'
    'NS+SFAspVDlHVyAzQKkSKwxX4gTb18YazinF+I9J1b39mvSv9xr/tDPrnW/1tuzHAubKO1cdgfEVek/i5NO6sYrIS3NCBxbs13BU'
    'FpLdLLtcZvjJMkb2PTeNWxpi727ztppb+GncltN7ZyJaqlwjzJH2SeaEfuQCNpeqXwWKkloCVi0IFZCcLWPUv01aLRpE/DkwvucO'
    'f6/l1HEfBpGoj1jDda0kd8OC8f2ncXR6YPzfW0trrs/TDvuGQuMmpew7g/ldTsOlpr5vEPYzY9RSJ2kuAAfUVJwYsfgk0Ys84hbh'
    'SLB2U5Cbm2e31eK6mwLOEVN3t1T/DuCp7t3KLwzpFkZ3Z1aE5F4ImPeZAPGZbBcV0j5dhDAMnKxybTifvJDLleOy61hOM8jgEopE'
    'y4kDrWgOgsLF5DRbCo2WXRTMQcl1mz3oAibJtAmstoh4/dYW8il9k4LzhEpVI4eV7AlYKKth8eqwWrOARWoDudZYprj2XZRW+Bvg'
    'aglfQazwBVqbm0yS8JuLwuruwK9JGGtZXaiZezBhAVGgalousTmw8vyijbUNEkZhflAbOheB5SegAcSghT4rOT7k9Az/UyCZvoiv'
    '6JvsPuxXr1jFA8JdgLYFBOcD4QRGsizSJR5qSPt+xCs1xSPQ7EqwrS0Lz0M8vRBZlmS0+ZGYmpzD5OF2hmqVrdKc1MuAUpzFZRBm'
    'Fv/IaSSI4RAWc3KpTaEcCx5N5EVmwczbtKzgCxrnZfZ0o0Ln3HyTnO8bt9DWndmmL1QbqR9JKmLLvDYRG35CZkxzWcz7e6aN5zAw'
    '8l65oVoSjeX2uKNvjwSy2qudYwJdzgKiHf5nrMP/yOkDAD/Rq7w+Pup1cRZ9aykCwLBe6+T05fuPp0CO4rpQKGyMpcezkk/MTKQR'
    'UC2PDiHU13yR3ey3OoCYjoAcENVYxeGe73f2FDKcTxlIEhZWsVeWoRmxuvN5CrE+DiWYmGY3h6i2sQD3A+obsbV6EVhBIPgbP3i5'
    'OKKvqIy1gMBSIwBpEkWWjZvbu9amr1kPfAeIYAEUDyNt96gTj2VN6iNu6Tj/2+3assPaHJ9y4aPrNAS9ZQ3oyxB7f0/T7fzMC3Oh'
    '79WI0nozVMT4ADsS9OUIyQkW2S1vWbhmUSijNUCzcYv/3jnGqyUwC+YfajXa/5q+EC39bwEjjPKv6gZ0n//P9lbT/2dre/ik/327'
    '83+H7TsXKJNI3YuMCj9BgjzYdzQ7kF6utEe8PHz3V1gn3fXKIytV69Wr14evD964WM09/vD+56N3B+8Oj3o1eArEIpnhebco3Bkw'
    'O+bZDOct5bwTxQ8yg12VvCgiNyV5JGWWnTKlmclsVSxz9A50JtZGWZVYMSIsMP0n9pNatf5r0/aFTGD9+h+Otne2G+t/e3u0/bT+'
    'v5H9hwym6PMHm52PFh0D1xbse+WJs7csLkDO+pN22Fz6CdJhMy670wsQLhZ07FPK0DlVj7wbkYGcDTJIcn5jJHPjl18QEgICmktB'
    's0IouaI97BDaH3/55bsN1vG8wEvJxwXBSc2G1BUlvYMU8Ct0/k+5Ic2fXnbTR24RGATUODh+rbx2QJ2GDJCAQedcBkmfrcTnwGAy'
    'r6AjAnWQnvugksB4eqQYCGwRS0APFlACGk5yPI7/bqPAsYMQmuQh5N+AFJyg0gL4APIyZniOn/tZOIPueGhMq6MbKmaJPNMxljFI'
    'wt9toH0aUovETyIQUmCMgMELrFkvDsik8SMWYfTpchaRImXEy8VMZOj6Q5btR/l5LaC9r+/2tYpFAqO+lVy07j1l7hvm/wGExud/'
    'W27W+HGFaixzURRpvr+5eQ6ceTlzQDnZ7K4GBVxUXcICq82DvfHuQOzNx6NZEAxeDLwx8J35tjeciy1/PBjOvRcv5qNZWT2HSQEu'
    'OQeKxPoPIeFm3fzCA3bnRnMEMH6xM9v2By/Gsz1odf7CC0Dw2Rpt786298Zid9uHjK0d398OvOF490WwtzV8sSt29vwXe6M9b2u7'
    'BK5TE/VMHzT1DlNPaMEiXR/iOSPMzcvS1Q3PiK6SaCknDTbGT0l2yZZg8g0+AAG5f5ImyRwBWCUjsKtO4BplUoLWNG8Qs1qSSueG'
    'SUih6AyWNgnXagfqp9WWylVpbfpA+OjeBSvfIwUeiOqH6+H1eAT/gf6WFyiRwFLI/7EU4jdiFAmUoPwavHnmLQSs6wSWXcjcKcee'
    'wJjxNBGWqu8tc1EuvX4IyxFduQBjxvEHwAssjvwSPQMFLFXvygtBV4hErZHDg5PXB6jnIEvqB2GOphGsGIdzkRfETMYvX746MH4e'
    '1ZmQ1GNrS/r9xzcfN7npNExrwoocUvKbiA1CHzkphr7BPaDBcleBH4e/SVUG5gC4CFnysiRFTJUmj6MBfpfqGbsS8UTRqa9qFl1G'
    '7jpdnJiuKrI6ZEeETi8nLtuv3C3xPE7RILIgptXa/gP7BkF0/eCXX1a6ONW9ATRchbErSSnfR3uHlgVIWZnHnhIuYpCygGGNtewc'
    'UB6UOUMtR7qQdWUFQDd8fNrOY4eMrpxZ6EH6LEnwGOKVF+VCywRMFTWPil2Z+Vk+FDgZaI2OHTmJdTchDZUNByIdk70u70hC5ET7'
    '3ijF+Jzwn0aecsqTf5u+SxKpE/WlkS89XfhPIw9RO8F/1ng7EYbJagt/VzuRhAvvXDzcWSROsgWdK5V4twiC1no4pz3ZmXuz3Kq6'
    'YhvfG0PR32s4w0oBieHqHsHHumkTUFBkwltU/iQGO0CLa88vohvHODCKJezHWgHfi3V4nu8vF8sIzz+I6mBG+inxOhLOpIOzWuGF'
    'hI/sD5AmvIIYsA4QhBYB/AvYgvjHMgQgzBcuhH8J4hdwuU8XImbDbMrSGnJxRA1IZAVWjG50eCHybUx2WqsZe6Xj3PlE7gFOvlxY'
    'I1v+Qec16b8mvdj0FVwyLs3nqrFKcB67FwFW7yLBqkMNYtSXRllm7SIZdJN/Vbu1ENruOJI4+/oCeKaNvfIYKbVFdxYl/qVVcVOi'
    'ojPkqUbtn2mvxrNojQC/OYG5x2nzIknX5J2Px7DQafQ/uwgz+IPu7rBlTpR0kNdcLmpwuvwnVmxVrbagfxNebBqeAPoPuEG9A+yM'
    'cNlj6UaBD+LNR0tPu6/pxvge1jQV/tKWaxh9aMtc+r6mcev3ro9hxxopd3jm/+OeouaR7mvdK91VOsQLEGs7ZYnTmo6K6mypbJJ4'
    '6Sn9sK8JnqUoijjxi7UO072OzfWzd9Uhb6ud5PmQ2QLM7WytnKWOmdrZ6ijQmqlmV2kVY1+bC9vC1oejPfjnxQ59s2VvOkGMukBw'
    '9QfDGH8ZDDxH/xqYhyaA+Y1lqw+fAOra42dAdXv05d3+NiSjOjz+wg4Tgd3X3Xt7wzqPWyRstuY+fUxzD40MFjEhawzMZzwCukG9'
    'fWKinhODmmzaj5fuUMvFc9xeLUu/qER7i5I5eHlZFVeQQl9tFLgh1CqMeIPSfb+Rd9fKjNV+pLW8ZG8a7pnvFY0JsdoXCbpQKBvv'
    'PbB0e1tcW7y1ldGktkT8xWTYPelqluukaNWXUv3n0CLM2HZb8Lmtt8uWCFP6lDY6ZZJBQWWyXzJ2FK+tIYHZ+iju/qlOA9ba/6vj'
    'ni86Abjn/G+4NWr6f+4Mnuz//yz3fzdqZuBOK7D09lx3jLj2NFCT70p/z5VHbroE+DnS2hphrSF5eP7ljH0MaxIob0j1shfszlV1'
    'UvMeo+ME2TFcXaUBh9w+2v3h8ljS1g0A6OfmxTfoVuQtRCEyYKCgJAMX5Ys/aKot81Dzrg3CKbNyy7Y7nD7LgnjMcZ8f2ZftiN0c'
    'W3VA7oNnkuNOy9J0gUWVRoRbVKLV1dsGrzZMqgm/6W+vYtcdF2Tsu3+vmBar+P9XYv0P8v/Y3R02z38HW+Mn/v8vxP8l034Ek/4M'
    'hkznyHlLt6hM1HzDcVyzJ5SqQhjT2Qo72/U0wzYqkDuPqHZAh9FX4uDqXNo1hnajp4SNcB4CI6beviGtwsJmhvYqNkpL7j6zcRdf'
    '1pqzNERJ9ujMI68oRIy9LL+X3mqr1n/b5+bz+cB98t9wd9SU/7Z3n+K//MvGf7kKMVCUw5TUctVyfx67Jwt00forWdxzlIoUsV2N'
    '3RzzdPNfy1frQfeLU1ggGZ0W1+WtR1rt+FQAmc7qATgvj14dfHxzys62qln2bMemGnYJdkiZdIzakq1N5F+7q6a22s/6FMWkYjAr'
    'Szlh7OKJC0xyjjzo9zBvlOKXEreq3kiZrmI/d0+hw/5J5D8/C9NC6vtphtSYL5JL8TUDgK3n/2PYAAat+59P/r/fzP/vA5rpYGUv'
    '/YIOT/p5cRMJ4+3p4bt30vEEhCB0GskpahVdUQGeip5qgUHUAku9ePDNUJWWnYNOmgv128+v1NcLL8dro+on/+lKcECj9dCfT+X8'
    'miex7sWmvif5ygupPeMY+nqc5OF17X7qxbIIo9W3VfOb6uKql6FPWPlTLNLab7zGqr7fpKJCgX81Ul/j5SK9Qb+mOOWe5n6Y3jjK'
    'OUmW+l8XyyiiWwrS5KKLbk55f1eNTv7e2IBZPDp1D9+/ffv6FC8wzv3haC62xXA+3NoZzQLhbY+2Bi9mo+E4GPjzob/nz3f87Zm5'
    '8fLo+PQnraa/EwwH85GY7XizFzsjbzYebs33hsLb8rzBwJsPgxfjYDaDmn89ev3jT6fuyU8HWG/7RTAe783mA380nA0HM7E3HM+H'
    'u/O9rcHWyIdMb3s2Hw/3xPbe3t729nw+mo38wdbQH23PZi+Gnrnx+t3Lo/+twPmzvb3hi9Hcn+9ubc32xAi6Jbwdb7SzN/SDF4P5'
    'cDDyRsGL3V1vNpx5/ngw2pkF8y1f7O5tQxVz4+Tt+78cuccHh385+PHoBK9T8mkixzaaR8mnyWTkDHeckdmTOfP+pci8XKYPtPQ8'
    'CheTyZCCYsrUReHHMaYNqjSaZEwb7ThbKhEm+jIs+rRDTiYDZ7RVVSAiIMC7zlAl4g0s/6qf3hQXSdxHFT2CSZ5Mtpwhtra3AwVt'
    'Ob6TN+9PaXADUnB27Y3j15BUptPRvJV58bmw9mx13y6/8CxcKVJ3ozNU/C3vlWUzvmjF/iMtlUSuXgfXgBuE5yIvLC6KphdygDRt'
    '50JcyzzVapR4gctCo0U7EYY6QW9YkcmOJLkj4qswS+Iz8/SV+/HkyH1z9OPB4d/cvxx9ODgxURgyh2ar7OHHlwfuz69PXv/w5sh9'
    'efTz68MjWZjLlgtXTb3D58TO1RDHWczZDQEXHE1rKdMii2R17o/G6SsD6eJPed1pjoZFXjWp8NnjxUiiQMVXhWyfGKNDYICrOOxD'
    'nZ8BYblEcE4kzj3/xoXfrNlR3ysmyN9crmdV1UC3a1e1H9GOw94f4W/sjruu1cofcW3zDYh81i8lUXRgTVBhJrSqkBvm4fHH/YHq'
    'NrolJjEaPFM0HBZz50f8amn3H2tlHC93gba8ZQR0tq95Br2BWWE3yzBGYV0GzoD2pcfA6athP4z7p69GjfnkcTuakx/ZMIlijU3D'
    'LCGh++9G8zIfriK8LuvivUardTvTkR6gVv18qUWgTJZ/j7soF11AZ8pYguX86O8xa2ftYuQjir6+RrUtu7kPC93s3dOH5iKpejN3'
    'kSE22paJa5qpRo+c0mUHjXwTGLBRS4AUuxZOK5SEQ7urVA1P4btVTYZWo0b7VYEpTaL8VStMkxbGucgK5KN4V5dmW7P6tG5Gimvh'
    '4zXZFJighZMv6yFXheFgNt51rojFdfGL69ZvMRLqvDQst/QP72r5SpeEdCsEUoeFOWG/UAQ9D68nXb2d45XmqNHjcqRpkloDu5Un'
    'UUbZGtrYL86WvlfIrjsngu3JtMnJqZDrRrF4XDoocSnTMxUtl1ANndUimt0UIrfsbtwy5AZi5R6luA2fyERlYXkLRLhaJ9Ql9EpM'
    'tmjV9FZBkYwmO5+hd9rVyPGvisMkwqulXA/TDt+/ef/B/eHHD6MPP/7AfUNBDi0NCqzDX1xKtwAcF5NnQZSqbb50kfVn9Lvka6wm'
    'sVHQ2wMqzrCAsc3pojTVu+5hyNNPPQMnYuFdWwQTQ8PdTDgKkYHBU/bp3zNzllyb07PR1HhWSxhPbfmV5xVvZqCkTyYONEVIyRlH'
    '0AP51vGyzLuxzq4x2upzbB46cQPfLqaSRq9gpYWMC4aA8eZUolU2oDaOVCMkSxXraVUjLw4WXnap1+3h/DggjmD4gzB/K4JXYVSI'
    'TLuMrSKSVWiypZVWkgT6z7ukCOUWIKBnSHC570UCpLaxpIQanqEgJV6QjQeSw4APwbDu2f5oKuWJv4Boz9enSIAAuYJ9XitHXa9o'
    'ut1SdGJWzbwshCXIcQY4egX0Qn7D4HAW4p7++WTbsPhGPcPCSaB/LjiJ++lFcxf7jn+pozg6oIFPXEv9vChrRGJeSIICdsmOy1EC'
    'JKk6YvQlVDnfGcVihhphbBE6ums9r9fiUDgrW7lRrVzI8rOkKICbcjMK+131ntfrwXqjEf15Ijua0GU2/M0g1y1C0AVBZiAcBbT0'
    'cP7NGn3BtghbHkDsSXjIzygNm+1xo6VoLiJkCVdhIBIUk/maClRIo5CJSbjs4cT7gC7CSz2VpfiqqpntszxP14OuhD4cjNIjkx0Q'
    'WNBREkFYc/NY1s831Td3uHlL3bhziuuCZBk8CAcuhvufxSJOPw/P1fCTTxSW26KwCHjjHs9XgEGSV3Nq2RwNQ4anwAyKipFfEWBY'
    '6tA9hxpEozAea9NUYXEFQa2l97EwCGUEUXj4vgEoT2JTCosyNAgvLrrrh0UY14ICcedyLVE1V1bj7nNsasPa6hkje4p90GaBrcCy'
    'EHfHv0gkczuzCFxPyaw99KiHrzzweh6Fqag3ToYYqoCZ0tloS7bCFIJ36+42VLiL7tYIq9yn/Vqc8Abmcb7qIsM8FFFAA8EpxD2B'
    'e47fsAZPJs6QZbqmPd1oRKawGMAZBqiVX4fVV9hXMKzSKhx1xrpgOqKwkzaGubLkECmhO8BEa83OTQayCHO6hbRv3OJA7sx2gAlG'
    '8hlmo/jIjbVKzYBeLzf0eEpdMShqPYC26f6VtK7BwiuWsA/L9WXcEk4ojBUhBb5x23d11sL9k6xDLELJOEhUUogseQeL13LfxOQo'
    'KfKJZkeQCPS9VMo1PyOwQy9FY59VimCSO+sSsYrIhWQC1XFTtkgIOjjGi6DH7qsPB2+P3MP3H9+d2nY9YgfW/N4YDu7FmU9WRhJ0'
    'MISJ4PtvDAFQR1JjcyKRpnuG9I48myKV6YFE/mi8RQpgRRGv1vrGS3FV7chxsqDoQOzJ7hivqMXXL/nSHUwR30F2assK8Ur+OYjf'
    '+qgUPI5BTwWfGRbjYHMTkNDQDgAaDdIFIQ9XKNlxJJAe7XMK4nNjG3ZJhvQc1kL38tGBIVK6VwzqjmG8FO2INR4HiKpN7fH7E57e'
    'k14Fv4/n4K0AOpeyBJIXgOL9o6ufOM/JJW7EeCGqk6KqZm1oTTVs4649cLYfObCVgW9I6qsk0JaG8IwXU3fomiwJcaRNIZKqsmS6'
    'MoiOVVF+T7MK248cVyiD5nJryDC1xlEWRRRzCQpriiVAgF/COt57IDc12fCFl3aNOfSzn4OcJQwCITG38FKzmxrIPiICd1GynHDB'
    'aZbp/JqemxL5a4hkXTffn8g+wrqIWE1i6LJj//P46MeOnkFvxiPZH3ZitrhrUtLRes263uu3H44OXro/fjj428nhwZsj+wymHpiN'
    'cj9f2X1qyUGvPxtnQt/NSEzBMjAfaZFaVNR+LAHoDE4e72i8TpouM4FfcmMm5om6S4wh69pu3bjvY9xFZrb48869VWtvfzAO7sx2'
    'JaQHV1oD5KWcTd6T0CZAKnhuwtcSJnyHJhg61SZS6Jqm1WAhczVQmv3hy26wVX8fGe6tq29fAEKSCFMexybDLbjqni1Zif2QJTDX'
    '1gABA9mihHRnrhkBlZaWGI3ynSKR1hl7HQop0nm+nM/Da1jSgYd6A0MkA6n599jkyIIA0iL3A+CRNoZJxqw2aNy1HC8IrJLjt6OC'
    'wZ6vHmi5NUkggnLmfkULeE6BhII+DCwbmbzyIEEJTSbuzlgA/twT3c5UfYHi6iuaNHlrhkS1Y98DRh7PLxfueGTS9XBe9XRZ1UbT'
    'F7meuDJAPz6CU3YYOffwzr5HNpViI2Joo9NQyNtyJECqKc9tMBaKMq7TiW5GSiOf7joH2fkSI3ocU47FgVAoWN3EdYPEV4Y5rolT'
    '53qyimX2+3i4i+/iYJBtPJ5dW5jNiw8tjVpan7U0E4VhDqmfF8DjXNjEMfFCROnEBD06MaQ6R4K3sVWpZSNzbSMqlkw/DZFe1raD'
    'oRdQbiTThIrekAI6iUbXt0O0oQ8djZHkHh5o/GNFZWWD/dz6zF7vrQ1VcjoDISD0B8HkVmlnwRiYOXIBT0UqZBWQkml2y3QtCCvD'
    'E7yV58s0jW4AhxQyhwkI905FHaWFUSnwmn58KW56FManZ5RSC2rWlqnQSx2Rh5P6oTrq/6aGRiqnfvcM/RRd36M5GA3a1KqgiHTb'
    'XZ51WGcYswbPP/qHJlu5sXu41DNx1aeRY+5PIF6YUy1Ypq0sIHVdiloDTqBGd69OdQsoueML+Bg2RIUzulUA9J1B4fQM6qA2zM1t'
    'VF50zBYk/nATfglzswnfaZI2MTAO/g/yxdyP8Zu8ko+5fXIN6idz+iLPeEM+quZipO9rTgeoaZWZjX0Go7pcmzZVKf0K1pnxpEDE'
    '3jCEDja4QrU+9FReiFenqzRKnsH1Ozub6CoLXRmT1aTzSbbgo6BHDZsa5Ujnyknt1FynzhoFlgusWkhaVHBaV0q8JqVLhqOVResa'
    'eW0ZPlwp5/iZFVCdclZqURw/oa1FSdmHNRMSj+/piMkzx/sb6iKxwZbZuu7RFKa0iUR65fZSjhTdUDw4uqvJTOH124Mf0bnk5MTs'
    '8egwU94O9K5ps8YUkN1JJXfjJMbtWrpsrwgdzb3fqOwWGE8ZPTql7eKMDWxkV8AzQc3TQ0292o5c2o7YOKlZeLj++lj0ZMIgMRpZ'
    'o3mKEJEHvRRXOoGW9se6xbpi8IDP0obsQJNmaYiiMpoNtcKANEUrsV4HhsI7GccoPhdDbExOWQRvAiWXNJKDn18bC0Gh0nA8t3oL'
    '0lrUPRn19VqCNk4PPsgIJ3gHqWdEIrZ47Mi1+VsZp5gq94y/tw0Ouqle71PbXN8rHc6cU4GnxF5285Kefkwy1BvxFB7SGwuEItih'
    'hXFCJi/Zw5Z1iVFDcVyahwOlzYoNNjUnOovrEQJXK7dcJszpRIEUXDYYE5d2ouQTemUjizYd7yo0qwKorsVsRCsH8kjFF91uIhXD'
    'G/GDB14EHf9pFWenkI6DEB4Eu0NRiLceg5auUp+kqxSIvSsiS7OfIbCB9AYBJrNfLQUIK/WMLeOZMRyM1J8OQ8kMg/p46sxVM/NS'
    'T3q6lZpwN62vtE677wq1CYaOapPepN2NrCbvZHHWQQfNCHQ8Ro+3KkL5r4CCuPlAgk6YWg+QDPQO7a9UoyRUGckbu+IEy0Wa6+NZ'
    'rVrSOWBcHztSJyYyzlZ0V3LpTp2znJSeQhHeU2w206G4MZUt4yiML7tCX6t1ATLAIrkSVtlQu2z55I98O0RjWUafOlNCs7tqN3jr'
    '7br6d5t69p3aJv7TuG1uNn25D+07Q3xfYn2sdMlS7uEGdWW3NjeSsGtzQ1/RoQRFCIygqE5lqk3DvCNq5LNFtOlT5NQWvqSzH23U'
    'OUjO6lBsMjHts8FU7qnCRxB1x9SpPM4h7x9QVioiYRHAVCNA24L82lPhJnEMUiiHNJaEZQhKyKpk5abhQRUlMbmqUErKPbRFyEPf'
    'MheF7TW7u91upNrh0UzR2PSlhYWWQ4gepLTCWkAUZk15Srff4RzuSCTQGuATSnWmqarftQBz8AqE2yRMDIbFpMmaY5PT0ZkdmmkW'
    'sBMTw6vbtDS+w/OKsiAGKpuMKo8DNZdrlJK5eUKng2GMjlGRKGDwt6qeEq+lWCpDrqJsquiLZFNanEBIdsV+mg9whHi1CvGFIbVh'
    'L3ZdpG3XNblrbP15us3zX+f+DxsRSUH4OreA1t//GW3tbm+34v9vPd3/+Vb3f0iD4zerKC5udbcHQ88ax0MOqcYaI8l2mooupG1C'
    'nh599iUgqgObRSSfrFeVpPM2unDW7vjc97wcvx23/p6NTMLXcT3ylE6DVXfdV8e2lhdxvGWAwaUD34WlM3QJk+Ug3iXxK7xaKkBH'
    'xnee8oskCnqMbjnyxvW7ckiVDt/TT4rJ1bDjChA+K1m1u+6xyI7KrADcH6Olt/L9hkddSqoufUjbQJYkha12FRkreULT7dABpEUl'
    'Vm6xa9zqy21VwT2r5CY641bp5EcgBQ8W7lH20GbBXmcl/Jjj6lG7sXxvlD0aypjyvIzy2EuBCgp1rwHFQTJGS3vKvk74FnbCliaW'
    'jpzKgA1bOUo12ih5T5827DJoZks+nVXC7nSj5dNFeZr31kbNzQrkBi5JWNoy+MVIAy8xUzqo9V7uhyEaIICzqLQgPMeXethzBdOw'
    'uxJQW7HNuJfl8dkUHYEwmTQO9hLrD6fUBSopT+mmK0G1D8im5CZWq01nZbZ9r23zdYxsEN8/x4i3hNfXLw3lOAfCGKXphk6e5zNq'
    'jU8Yp9MzKjVV2iHkyQgRMoB444xCau5Sb0KVg2A69MqlbgctTzAmxk6d9HXBe8o2wPFGQ8tV+pt2WIAju0VhETQWDVmS7qgvXMvh'
    '6L8WqHdQ/Vb6Jd49CJ9y6ShrGq8WaaC7CHVbrULPGRXFA4dbJIsm2Z6NpnZdR+NO3tV8sqSq1xhHC6elHQT1PEtzk4fEfaYwOhae'
    '2i1XxzMtV1+rCG7ayWm6HxnLaHpumwujAfGutEzoEDsMFOunAb1/loBKn0I884EkPfxYo2w8hlGTIfnX1PjvWhryrek6xlmJIOhe'
    'B1sbiCGqesNZmqhd3zyoW7Qz0PxJh3Q5VDmvbK9uMb2eoa9E9bD1PLxmJxYepEu4/pPC9Z+mmh8L+zIh3GSF0wpTHZmnGXTLZaU8'
    '92jAUB4qHRBa/ikrjnCoJeWPqR61bpThE4cVfkobXWc+5P/O0LQUvq9TOY79R/l8NqWt1WPfSrsKbsSwb0ThOZ7RfPjxh00ZE0eN'
    '3FSe/Nfl3VX0C6FbED2ejU6Pkko0sOse48oTvdMRTxuSoqoWtqX/FyUpr67R9g6wvO2dLszL8pzW6QaGq4muVKCXlzOmVYGMTWGT'
    'fD3KHa/mEzJdi2X1fEUKS2tBUbPPMThKA7dyncmTN25U3T2gLcMtz5AaKw+E0iw+V5ef6IFf2i2w85BBD7KeY1yi0uhgS5dwOV6j'
    'VmoISBluo1l+OKjQXp0Trlv80u3/hP1/I+SNOFq+Dtn0b8OJls/OqMf+3i0XxzfE6ehlH9aFa3OPJ3dRmPLEOyBb4y03SCRShLk0'
    'ntdH098awB4IWyEHv4YFN0Aa2babUPkuCuSg5y+nb1JB49kzArmEbTzJFtbA2QYEOdu23dG8oiSsAEMPEiAaEJ5qnq84PLdyG64j'
    'n6KW1rcLYMpez6AbSHKuqq4MoAL8i71Eyn/2bNRrd3ZU39KaF6AyXHqWhc0AFGrHfvYMauJdLS1vs5ZXf+gTL9HI60e18Yy2d/E6'
    'EDsK9LryqBd2O3r/GcDch/+fq+4i+H385zlVmdLtemfrxXALUbC1NxrR362d7QfNQoOquU18dmC/j3chKL38TbVu8N5ca3w7krQk'
    'kvEGU2tF9YztgY2O5KOK6s7kBaqbPlax92+e49+euld1LZOvKXlaPmZeMjUgOpSjQTlDmlHhlLtIsSZiI3OrkCDZ4WBKtx8RE0Tz'
    'zmCj9RQBPmPjZuczGVJIhSNCxdMl5V4ySuWd1llLv6BpVSiXWO5azn1jONp1mB2N9vRutdrHG445HTyeL5Nlzrcd8bIhMM44T5Nc'
    'WCNa/kO7jLjA0gXRtYzbhlPPfoHQgDpakscc3CQenKJRQcV58+iFV6mItYposKqStXTje5hcu+lO0RbWSETzkwx3bBWC1OB21H1m'
    'suDQY8p03/JKSCMRbDwNeY59IqoQdbS/UZBnRLXePcDU/5CjimW4cPJ7sWzenNCYApLRlaXu4tKmwJJ8z5ihmzPiFIMKKCMDXRDV'
    '4p1Kq81EeVI0fHJOyDyGwuxDjvXrjDagO0yWpmfYyHoFvXiCh27y5Kiuedi91rUlKYVDH7s3QLx12tIv2mwtL6OB54XnX1pnrXUi'
    'r3IterxJcLvTJr8tvKgCxbEBrLPK2UVKK7TVKCZXAYMU8hiUS0hnGBX54ZKopkTHKd0nRpJCkL+pzTiXyh/NOPEH7GMtsUMPUvyX'
    'w5XRak1kCAq7ikXr+Omy851h7KTS4LvWMvaycXbKxNZ5KKwOHsmOID1924o33Wlb6zPcCHSLcd+5WzbHKLeJzqzm8XLzPDjQLvjT'
    'ok0DB1336LaTxeNQjIyDFzM90jPjjMOeIdLEv6gWYi5EUDE1zlTWHSqCP9ZerOUqyGS4vPJNVCxH8hkUunNRgER3Su/UwzbY6XD3'
    's8jC+Y0yOOhPtSG/Y+u27t9UmuxaFsy2YUG30yjbyH6bnLuNKN0vl2uru/usezV3kNfz0BBWOWkx95H2KaXEy0knLyHY1ViEcaT9'
    '0YV0i6ZR3sZGEgPELb3IxWQtT0WpaIfRnmBobm2pbSinHTRX6zU6LNhWq2KSFuECWFhWciRKwYitC0vGItDCYAMXzyZD0QfBTR54'
    'B8L3bibbor+tNgR+ALQJMMrcMss5KUT65oNVNo5+uiLlV262B7DPn3uLBY5UAk0e4RV6ni5JJMIxOnQDC49g/WXglR6wUEQTpHih'
    'Qz5qmqAZpsK7dBdikWQ3LnD6Iq8hDJQb9BKr2OsDfP9o4VVqw1AtbbpW2HBQlbHMNU9PcQ41V0CX7lDzeUQ9QNk1FdliyYdGlk6y'
    'thYVnE48ahtE0zePzUvc8Vv6c7fJf/M7uZfXgN+7o5fGWERXhYsBw1JjsCW361jC1cKTZc8I1D79+5xqtY3YvL0ppRxr46b3zFqn'
    'lzdWtDQ2s6beZRbtcPApRX1dXpB9aC3Bmq1SrQjSEChKvYU0WSTo7ypWuPJUImS5HesNMxLszu25Bcub5figpHLKo4faFrlwkWis'
    'qiGFUjSDSUbTldvhg4WX8cpWYAVQM6u8kxoyOdZ++J30d0ncl0I2NdskaLO7exRWn8K7rpsfZFhWN4DKN+rMUgOVYoNydeRfNsaF'
    '0cTi+/3MePU8766EAkrO+w7JKtyXnuEBl5xo96AlC6vkKEIHyDz0tyedtl2YdVM+CGERaJClcJ2r+e4oMJyuka7W+wcRmwPoPFAs'
    'ZNUYAwyahRa8B40KOf26a3qTa55Ccph03NXtCldtSNpsoiUfJsyVve3gvJU3E9u/cb8gTlI9F4S7ycK7VtsIaI8Ys04EagHKXagK'
    'eCzDXJS3JOgCJtq0qzQnLUxNbsi9KyF3aNylBMVsQgqranRKbR/oaqpWqnVHgo6OXQ0oN4ipVlWtR/cZVSy+iQlyvqmkgtxN4uiG'
    'sa1CDl259+yUUIRC5XQqpYop07lIUzetWlg3Z1oXVAVgia6fX1mJuqxwJd0QHEg12cnsWhcu/mgckMuFl2L8A3b+uEDXiASvUc6X'
    'kfHh/aFRKIcFtkTn0hkRA1xf49FQiG+FbpS3ya8cAuTEyzj8xxL9uuumHhC+stDns26clSWuDpP8Sby4kD02tUc6eElWgEFA93Hx'
    '8gFePZpGOwwbv9lO80TVERCh2t5oXKnvdNKoQ+vuO+xlLnNmt8SVebcm3kcFpuwebmFAC+gHa5ZAXDIqBXTor5XsyJ/eSfUsB8Gc'
    'uof0hp1T1w5NlMYpJrfAu6dEcvBTkp7JtIf5dCREhNhyiYQ1XumSyBTwEJ0Lk4PgSn6AN1OxVlnbdaVLpuu2WuHwplAUNTe9YP2d'
    'c1e+qF6+577KJ+VMf/Z92mpOF/yk63VTFsSVVCuA5CPT2ZLh4gPwfEtzv3Xq2mpS8nIoKr8xLPgtKaO8yVttMg1GLluv8rVfbR/Z'
    'kqe7MC0V/+ZL23jxuSzQ03l03bdX48StFlg9dpXaXK+4zmlHetBKlkXE6jIV3+c4i2U0x1nptd9xnWBDd1In2jj9cPAaKISmCA88'
    '/2bcQgfuHuv5+tXvOksjw+97CXdFbdaGVG3aEaWePxmvrcic5PH1iB91VdtaX09yqp4KWTWxWB1GygX+ZFeAOP2hV4/ZdZ2AVzp2'
    'GWRDY3ph7npXXhh5M7y1tOoa8gndejM84zCBksaPxx+NbBnjRm7QO10gwauxGNTvyirD/uRMDI0rOtwNrmZpPbZlOWVkox+St9N3'
    'tsU8zP+X6sH3ZCa+hf/vcLQ93Gn6/27tPL3/8q38f98KL4cNlb18Xx2PR5WzIogayfL8goL/z/FZudInw+BgBRy6+gsj/z/IoRfl'
    'LdgtYXf8DBffR7vzdrvDqzqaybfX7RXx/9U/9/FeuDLyhJsmSaQMR5WB/o/G4Y0fCUUN6OTZjlHBNja2JpWBgkCwKCiMHeFIhlAs'
    'bV8ty3lppnqE3ZlDoWIsRTohQQe7ac+uBVpT4e3VMZQytHcfuZWNO1Jdt6iJTvXzJV9rx13CW57jFmUQ3+QHM3JpW+T61Q2bFUZF'
    'iaS6KbM6O5QW/+rEkMC2DwgVrqVB5B6nHbYEPvT8Z4UFUjZpt4/Faza7XqcFUUWRYS7Eu4+FhFgqxp+8bLFM2aye96QKUzvJ/TqH'
    'DzVr9Vc/S/jsU43f+QwgWGbl3eHOc++5SZsCG6xu2VJlqlnBMIU8LaWxa80pOBatVmQdQoOMQbKXUVJ5PB7XIQjPFGXwTzx4UGm2'
    '8d9ozSEBoZVgY70B3eJTtjPZGvvRchpdDwQojVPbutkHON/cpRAOeFe93Vivsp7Lw+lqcst+rLKhP95+3mE7X/Waqma/Xm8TB/Q+'
    '2Ci+zuK81tq8xkIuzcsPcl1uGsnn5WmyottWs7Wl07GE8pvYh30vRq/NrklCd00kwj9P5Iro8C1QK0zx5A7Sqcyw63lxSfDM22A6'
    'PJzrSjpyOM0qG61x5duW5aeyYyWXqERxdZeb0wwMnN54E0KzfnBJBRi0eVlB8QXdVqG61oBGfKptnfhcK3QXdGSI2dW9wFWxlbDv'
    'SsEJdnvd3YFGjzzm/k1rpWfCfQz/fh6U8oOHq8Q6+ZZKzby8ihyb98+VYXGejkeuskoRGsy1Zr41hr2mNanTGPkl5scOQ6ZLbkFd'
    'ZKBBhWGl+JyAyCsRgaoxmdzfUDEHJC28YrGMykaQPQITyBXRYSZ65CWfXCz+QKBQOY47Ycbxo6DV7cJMtW51bqXI2EQiCbQMpuuW'
    'qbHSCrUFXxktfQ9PqaROVYt3oTbsFsjHGxWVfZtgn0nbOHpdPC7YJqAOK1FO3naJfBVG4ojyePeBspWTBDNBkCBkR/Y31m05GvVR'
    'VCqX0GSteV6G3es/U2Zunnxozb9fFu/nb4m6aVRdra7dQ5aAxLlcH9ohCDOQs3Je6tfdVp56Ym73aSfMHj2MR0eLjTsfBAALmK3y'
    'D43DcI85WYeo3iwq518PufDDwenhT3gY8cMRm5pNuU/8+1qaFQnott8YjZQT87lZWW/LZ7lHPaO8SrMqkicRe6c1ebTeCI2ro7Pe'
    'zjczXtNJs7RVf67xeqGsxhK5tnKl5KgqrGfpSTTwuotl3X7NDAx92KQs0wcIkvv3yOBBG4JKWuV9+Tsa1h9oUmcJbbVJXUdbT8dX'
    'T8OUZkVfY4T/V4yCouz/nfELvtIjwPe8/741GO427f+j0ZP9/xu+/+tF/jLCO77ohxNIK668DeEV+pU5OiSQzggJPoGB3tGlR8Lv'
    'exLw0Bge/H7uZSS8LHaUs4UsmMHe6C9Bt4Ndl5+X7/T70N5XkJwIA6VUaEDHmGyJ8aBDkLzxbYO4Mj3U0aFdVWaM6k+rqucu0oDv'
    'qKL/TloGrqYncuhBBnU5AJhk7Fl29SRAI7u8rR2oQhuN2+MY7ZEr8ZNLdEUfLyPdqatDcVpZYORzENhveW3IXnvHlAL/31XXgvBy'
    'sMQK+xeV4azZcz8pLjhaQuNaELWrrp1Wfjuw0xTloYQ2HWSFx6ng653zMIPdCOeI/c9ioAT05i8S49XxB7R4vfvAZxHzFB+0p3/U'
    'pOV0FVFSCfkdyXeQShekHu5yLqVOpMpeOUlNpMNSBe+Mnrw7p9dY4Nsst6BBEDChbfT/cwa2Pa297KdPQAmmhfduuqXbWerNvxuK'
    'sOEZTbo0ZYQJetYQt2+r+VBADbvlcke3MHr2N4Qd+M8gHF+JGKDSoZBcb8AREO9U8M+T6omICC+BqedF0INLo0FH/MPCF7Sky1dl'
    'o/QKPClYV23QVS0Tv6rIF/S0GMbsjHRUymtJVTs+6kFlBW7WOS/WVJmlPh6+ePiHghbLJjdJmcUm7V4Fl1MZbjOeL+hTBIWsNKSL'
    'ewzbpDZQzZK/vVopi9t+ztn8hF/L9GhiT1zVO4waKL/2ZBbZM6QKzr1uw+B+u2o08k2BlKHITB2OHOfdRt3y2OnXVlThkKQPFBOk'
    'dLHCBV+m0eq/+yavDtAdv89Sb6jHXxgJv6UHMwvU9xAld5ITdiOZ8GTXogo+hfF7+jx9nj5Pn6fP0+fp8/R5+jx9nj5Pn6fP0+fp'
    '8/R5+jx9nj5Pn6fP0+fp8/R5+jx9nj5Pn//in/8HV+gZywDwAAA='
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})


In [ ]:
#@title 3. Khôi phục hoặc dựng 48 khung MTCNN–PRNet
def prepare_prnet_environment():
    global PY, PRNET, RENDERER
    run("install-uv", sys.executable, "-m", "pip", "install", "uv")
    VENV = WORK / "venv"
    PY = VENV / "bin/python"
    if not PY.is_file():
        run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
    run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
        *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))
    PRNET = WORK / "PRNet"
    RENDERER = WORK / "PRNet-Depth-Generation"
    for path, repo, commit in [
        (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
        (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
    ]:
        if not path.exists():
            run(f"clone-{path.name}", "git", "clone", repo, path)
        run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
    WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
    def weight_matches(path):
        if not path.is_file():
            return False
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

    if not weight_matches(WEIGHT):
        partial = WEIGHT.with_suffix(".part")
        run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
            "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
        if not weight_matches(partial):
            raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
        partial.replace(WEIGHT)

import json
import zipfile

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

archives = {}
for name in ("Train_files.tar", "Dev_files.tar"):
    path = DATA / name
    with path.open("rb") as stream:
        archives[name] = {"bytes": path.stat().st_size,
                          "prefix_sha256": hashlib.sha256(stream.read(65536)).hexdigest()}
worker = (WORK / "scripts/cdcn_prnet_smoke.py").read_text(encoding="utf-8")
key = {"profile": "p1-3-videos-per-split-8-frames",
       "protocol": sha(DATA / "Protocols.tar"), "archives": archives,
       "preparer": hashlib.sha256(worker.encode("utf-8")).hexdigest()}
key_hash = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:20]
cache = RUN.parent / "prepared-cache" / key_hash
cache.mkdir(parents=True, exist_ok=True)
assets = (WORK / f"assets-{time.time_ns()}").resolve()
cache_meta = cache / "snapshot.json"
if cache_meta.is_file():
    with Progress("Restore RGB/bbox/PRNet from Drive"):
        meta = json.loads(cache_meta.read_text())
        digest = meta["sha256"]
        if meta["key"] != key or len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest):
            raise ValueError(f"Invalid snapshot metadata: {cache_meta}")
        snapshot = cache / f"assets-{digest}.zip"
        local = assets.with_suffix(".zip")
        shutil.copyfile(snapshot, local)
        if digest != sha(local):
            raise ValueError(f"Snapshot verification failed: {snapshot}")
        with zipfile.ZipFile(local) as archive:
            if any(not (assets / name).resolve().is_relative_to(assets) for name in archive.namelist()):
                raise ValueError("Invalid archive path")
            archive.extractall(assets)
        local.unlink()
else:
    prepare_prnet_environment()
    run("prepare-48-frames", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
        "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--training-pilot", "--output", assets)
    with Progress("Save RGB/bbox/PRNet to Drive"):
        local = Path(shutil.make_archive(str(assets), "zip", assets))
        digest = sha(local)
        snapshot = cache / f"assets-{digest}.zip"
        partial = cache / f"{RUN.name}-assets.zip.part"
        shutil.copyfile(local, partial)
        if sha(partial) != digest:
            raise ValueError("Snapshot copy differs from local ZIP")
        partial.replace(snapshot)
        metadata = cache / f"{RUN.name}-snapshot.json.part"
        metadata.write_text(json.dumps({"key": key, "sha256": digest}, indent=2))
        metadata.replace(cache_meta)
        local.unlink()
shutil.copyfile(assets / "smoke_summary.json", RUN / "preparation.json")
print("PILOT_DATA_READY", snapshot)

In [ ]:
#@title 4. Train CDCN, đọc lại checkpoint và tính điểm Dev
EPOCHS = 3
BATCH = 3
SEED = 43
result = RUN / f"train-{time.time_ns()}"
run("train-and-dev", sys.executable, "-u", WORK / "scripts/cdcn_prnet_train_pilot.py",
    "--assets", assets, "--output", result, "--epochs", EPOCHS, "--batch", BATCH,
    "--seed", SEED, "--device", "cuda")
print((result / "pilot_result.json").read_text())
print("RUN_DIR", result)

In [ ]:
#@title 5. Đo batch 8, 16, 32 và 56 trên A100
PROBE_BATCHES = (8, 16, 32, 56)
report = RUN / f"batch-probe-{time.time_ns()}.json"
run("batch-probe", sys.executable, "-u", WORK / "scripts/cdcn_prnet_batch_probe.py",
    "--assets", assets, "--output", report, "--warmup", 2, "--steps", 6,
    "--batches", *PROBE_BATCHES)
print(report.read_text())